# Lab 01-08 · Sequences, Series and Taylor Expansions

Companion to chapter [01-08](../../chapters/01-08-sequences-series-and-taylor.md). Handout: [README.md](README.md).

Run the cells in order. Cells marked **🔮 Predict** ask you to write a prediction *before* running.

In [1]:
import math, shutil, subprocess, sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from scipy.special import iv
sys.path.insert(0, str(Path.cwd().parents[3] / "tools"))
try:
    from plotstyle import use_style; use_style()
except ImportError:
    pass

## Part 1 — Taylor polynomials and the Lagrange bound

For $e^x$ at several $x$, find how many terms reach $10^{-6}$, and compare the actual error with the Lagrange bound $e^{\max(x,0)}\lvert x\rvert^{n+1}/(n+1)!$.

In [2]:
def exp_series(x, n):
    return sum(x**k / math.factorial(k) for k in range(n + 1))

for x in [0.5, 1.0, 3.0, -3.0]:
    for n in range(1, 40):
        err = abs(exp_series(x, n) - math.exp(x))
        bound = math.exp(max(x, 0)) * abs(x)**(n + 1) / math.factorial(n + 1)
        if err < 1e-6:
            print(f"x = {x:5.1f}: degree {n:2d} reaches error {err:.1e} (bound {bound:.1e})")
            break

x =   0.5: degree  7 reaches error 1.0e-07 (bound 1.6e-07)
x =   1.0: degree  9 reaches error 3.0e-07 (bound 7.5e-07)
x =   3.0: degree 16 reaches error 4.3e-07 (bound 7.3e-06)
x =  -3.0: degree 16 reaches error 3.1e-07 (bound 3.6e-07)


Note $x = -3$: the series needs as many terms as for $x = +3$ to reach the same *absolute* error, although the answer, $e^{-3} = 0.0498$, is 400 times smaller. Its terms alternate and grow to about 4.5 before shrinking, so most of the work goes into cancelling large terms, and the *relative* error is far worse. Libraries compute $e^{-3}$ as $1/e^3$ instead (Your turn, question 2).

## Part 2 — Convergence speed: two ways to compute π

**🔮 Predict:** how many Leibniz terms to get 6 correct decimals? How many Machin terms?

In [3]:
def leibniz(N):
    k = np.arange(N)
    return 4 * np.sum((-1.0)**k / (2 * k + 1))

def arctan_series(x, N):
    return sum((-1)**k * x**(2*k + 1) / (2*k + 1) for k in range(N))

for N in [10, 1000, 100_000, 2_000_000]:
    print(f"Leibniz, {N:>9,} terms: error {abs(leibniz(N) - math.pi):.2e}")
for N in [1, 2, 3, 5]:
    p = 16 * arctan_series(1/5, N) - 4 * arctan_series(1/239, N)
    print(f"Machin,  {N:>9} terms each: error {abs(p - math.pi):.2e}")

Leibniz,        10 terms: error 9.98e-02
Leibniz,     1,000 terms: error 1.00e-03
Leibniz,   100,000 terms: error 1.00e-05
Leibniz, 2,000,000 terms: error 5.00e-07
Machin,          1 terms each: error 4.17e-02
Machin,          2 terms each: error 9.96e-04
Machin,          3 terms each: error 2.84e-05
Machin,          5 terms each: error 2.88e-08


## Part 3 — Distortion: Taylor predictions against an FFT

Drive an exponential device, $i \propto e^{A\cos\omega t/V_T}$, and a cubic amplifier, $y = x - 0.05x^3$, with sinusoids. Measure the harmonics with an FFT.

**🔮 Predict:** HD2 of the exponential at 3 mV, and HD3 of the cubic at 1 V.

In [4]:
VT = 0.02585
N = 4096
th = 2 * np.pi * np.arange(N) / N          # one period, 4096 samples

def harmonics(y):
    Y = np.abs(np.fft.rfft(y)) / (N / 2)
    return Y[1], Y[2], Y[3]

print("exponential device")
for A in [1e-3, 3e-3, 10e-3, 25e-3]:
    h1, h2, h3 = harmonics(np.exp(A * np.cos(th) / VT))
    x = A / VT
    print(f"  A = {A*1e3:5.1f} mV: HD2 {100*h2/h1:7.4f}% (Taylor {100*x/4:7.4f}%, Bessel {100*iv(2,x)/iv(1,x):7.4f}%),"
          f" HD3 {100*h3/h1:8.5f}% (Taylor {100*x*x/24:8.5f}%)")
print("cubic amplifier y = x - 0.05 x^3")
for A in [0.3, 1.0, 1.7]:
    h1, h2, h3 = harmonics(A * np.cos(th) - 0.05 * (A * np.cos(th))**3)
    print(f"  A = {A:.1f} V: fundamental {h1:.4f} V (gain {20*math.log10(h1/A):+.2f} dB), HD3 {100*h3/h1:.3f}%")

exponential device
  A =   1.0 mV: HD2  0.9671% (Taylor  0.9671%, Bessel  0.9671%), HD3  0.00623% (Taylor  0.00624%)
  A =   3.0 mV: HD2  2.8997% (Taylor  2.9014%, Bessel  2.8997%), HD3  0.05607% (Taylor  0.05612%)
  A =  10.0 mV: HD2  9.6114% (Taylor  9.6712%, Bessel  9.6114%), HD3  0.61777% (Taylor  0.62354%)
  A =  25.0 mV: HD2 23.2875% (Taylor 24.1779%, Bessel 23.2875%), HD3  3.68270% (Taylor  3.89716%)
cubic amplifier y = x - 0.05 x^3
  A = 0.3 V: fundamental 0.2990 V (gain -0.03 dB), HD3 0.113%
  A = 1.0 V: fundamental 0.9625 V (gain -0.33 dB), HD3 1.299%
  A = 1.7 V: fundamental 1.5158 V (gain -1.00 dB), HD3 4.052%


## Part 4 — ngspice: a real diode model and a feedback amplifier

In [5]:
if shutil.which("ngspice"):
    keys = ("hd2_1m", "hd3_1m", "hd2_3m", "g1", "g2", "g3", "e1", "e2", "e3")
    for net in ["diode_hd.cir", "opamp_gain_error.cir"]:
        out = subprocess.run(["ngspice", "-b", net], capture_output=True, text=True).stdout
        print(net)
        for l in out.splitlines():
            if "=" in l and l.split("=")[0].strip() in keys:
                print("   " + l.strip())
else:
    print("ngspice not found; see SETUP.md. Skipping.")

diode_hd.cir
   hd2_1m              =  9.66983e-01
   hd3_1m              =  6.23313e-03
   hd2_3m              =  2.89950e+00
opamp_gain_error.cir
   g1 = 99.9001
   g2 = 10.9994
   g3 = 10.8803
   e1 = -0.0999001
   e2 = 54.997
   e3 = 1.08803


## Your turn

1. In Part 3, make the device differential: $i_{diff} \propto e^{x/2} - e^{-x/2}$ with $x = A\cos\omega t/V_T$. What happens to HD2? Why?
2. In Part 1, compute $e^{-10}$ by the direct series and by $1/e^{10}$. Compare the relative errors and explain them.